# Credit Card Default — standardized 10-seed protocol
Uses validation-selected augmentation strength. GAN epochs = 300; RF trees = 150; MLP max iterations = 300; seeds 44–53. Other model settings are retained from the corrected notebook.


# Credit Card Default — validation-selected revision protocol

Stratified 60/20/20 split; 10 seeds; train-only preprocessing; four strengths; ROS/SMOTE/CGAN/CWGAN-GP; RF/MLP. Each classifier is fit once per method/strength and evaluated on validation and test. The **best validation Macro-F1** determines the selected strength independently per run, method and classifier. The test score is reported only at that strength. Full grid retained for sensitivity; synthetic-only fidelity uses validation reference.


In [ ]:

# ============================================================
# CREDIT CARD DEFAULT — REVISION-READY EXPERIMENT
# ============================================================
import os, random, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, LeakyReLU
from tensorflow.keras.optimizers import Adam

from google.colab import drive
drive.mount("/content/drive")

# ---------------- SETTINGS ----------------
DATA_PATH = "/content/drive/MyDrive/CGA Ex/CCD.xlsx"
OUT_DIR = "/content/drive/MyDrive/CGA Ex/revision_protocol"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_SEED = 44
N_RUNS = 10
LAMBDA_GRID = [0.25, 0.50, 0.75, 1.00]

TEST_SIZE = 0.20
VAL_SIZE = 0.20  # fraction of full data
LATENT_DIM = 64
BATCH_SIZE = 128
EPOCHS_CGAN = 300
EPOCHS_WGAN = 300
N_CRITIC = 5
GP_WEIGHT = 10.0
FIDELITY_N = 1000

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ---------------- DATA ----------------
data = pd.read_excel(DATA_PATH)
if "ID" in data.columns:
    data = data.drop(columns=["ID"])

TARGET = "default payment next month"
feature_names = [c for c in data.columns if c != TARGET]
X_raw = data[feature_names].to_numpy(dtype=np.float32)
y = data[TARGET].to_numpy(dtype=int)

# Discrete/ordinal variables in the UCI credit-card data.
# Generated values for these columns are snapped to the nearest category
# observed in the training data after scaling.
discrete_candidates = {
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"
}
DISCRETE_IDX = [i for i, c in enumerate(feature_names) if c in discrete_candidates]

# ---------------- HELPERS ----------------
def split_scale(X, y, seed):
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed
    )
    val_frac_within_train = VAL_SIZE / (1.0 - TEST_SIZE)
    X_train_raw, X_val_raw, y_train, y_val = train_test_split(
        X_train_raw, y_train, test_size=val_frac_within_train,
        stratify=y_train, random_state=seed + 10000
    )
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)
    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

def discrete_supports(X_train):
    return {j: np.unique(X_train[:, j]) for j in DISCRETE_IDX}

def postprocess_credit(X_syn, supports):
    X_syn = np.clip(np.asarray(X_syn, dtype=np.float32), 0.0, 1.0)
    for j, vals in supports.items():
        # nearest observed training category on the scaled axis
        X_syn[:, j] = vals[np.abs(X_syn[:, j, None] - vals[None, :]).argmin(axis=1)]
    return X_syn

def mean_marginal_w1(X_real, X_syn):
    if len(X_real) == 0 or len(X_syn) == 0:
        return np.nan
    vals = [wasserstein_distance(X_real[:, j], X_syn[:, j])
            for j in range(X_real.shape[1])]
    return float(np.mean(vals))

def target_minority_count(y_train, balancing_strength):
    counts = np.bincount(y_train)
    minority = int(np.argmin(counts))
    majority = int(np.argmax(counts))
    n_y, n_max = counts[minority], counts[majority]
    target = int(round(n_y + balancing_strength * (n_max - n_y)))
    return minority, majority, n_y, n_max, target

def actual_alpha_y(n_real, n_generated):
    return 0.0 if (n_real + n_generated) == 0 else n_generated / (n_real + n_generated)

def effective_pi_after_binary_aug(y_train, minority, n_generated):
    n_total_aug = len(y_train) + n_generated
    return (np.sum(y_train == minority) + n_generated) / n_total_aug

def get_classifiers(seed):
    return {
        "RF": RandomForestClassifier(n_estimators=150, random_state=seed, n_jobs=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,64), max_iter=300, random_state=seed)
    }

def evaluate_both(Xtr, ytr, X_val, y_val, X_test, y_test, seed, minority):
    """Fit once and score the same fitted model on validation and test."""
    val_rows, test_rows = [], []
    for name, clf in get_classifiers(seed).items():
        clf.fit(Xtr, ytr)
        for X_eval, y_eval, rows in [(X_val, y_val, val_rows), (X_test, y_test, test_rows)]:
            pred = clf.predict(X_eval)
            rows.append({
                "Classifier": name,
                "Macro_F1": f1_score(y_eval, pred, average="macro"),
                "Macro_Recall": recall_score(y_eval, pred, average="macro"),
                "Minority_Recall": recall_score(y_eval, pred, labels=[minority], average=None)[0],
                "Balanced_Accuracy": balanced_accuracy_score(y_eval, pred),
                "Accuracy": accuracy_score(y_eval, pred)
            })
    return pd.DataFrame(val_rows), pd.DataFrame(test_rows)

# ---------------- MODELS ----------------
def build_generator(feature_dim):
    z = Input(shape=(LATENT_DIM,))
    label = Input(shape=(1,))
    h = Concatenate()([z, label])
    h = LeakyReLU(0.2)(Dense(64)(h))
    h = LeakyReLU(0.2)(Dense(128)(h))
    h = LeakyReLU(0.2)(Dense(64)(h))
    out = Dense(feature_dim, activation="sigmoid")(h)
    return Model([z, label], out)

def build_discriminator(feature_dim):
    x = Input(shape=(feature_dim,))
    label = Input(shape=(1,))
    h = Concatenate()([x, label])
    h = LeakyReLU(0.2)(Dense(64)(h))
    h = LeakyReLU(0.2)(Dense(64)(h))
    out = Dense(1, activation="sigmoid")(h)
    return Model([x, label], out)

def build_critic(feature_dim):
    x = Input(shape=(feature_dim,))
    label = Input(shape=(1,))
    h = Concatenate()([x, label])
    h = LeakyReLU(0.2)(Dense(64)(h))
    h = LeakyReLU(0.2)(Dense(64)(h))
    out = Dense(1)(h)
    return Model([x, label], out)

def train_cgan(X_train, y_train, minority, seed):
    set_seed(seed)
    feature_dim = X_train.shape[1]
    G = build_generator(feature_dim)
    D = build_discriminator(feature_dim)
    opt_D = Adam(2e-4, beta_1=0.5)
    opt_G = Adam(2e-4, beta_1=0.5)
    bce = tf.keras.losses.BinaryCrossentropy()
    X_min = tf.convert_to_tensor(X_train[y_train == minority], tf.float32)

    for _ in range(EPOCHS_CGAN):
        idx = np.random.randint(0, len(X_min), BATCH_SIZE)
        real = tf.gather(X_min, idx)
        labels = tf.ones((BATCH_SIZE,1), tf.float32) * float(minority)

        z = tf.random.normal((BATCH_SIZE, LATENT_DIM))
        with tf.GradientTape() as tape:
            fake = G([z, labels], training=True)
            d_real = D([real, labels], training=True)
            d_fake = D([fake, labels], training=True)
            d_loss = bce(tf.ones_like(d_real), d_real) + bce(tf.zeros_like(d_fake), d_fake)
        grads = tape.gradient(d_loss, D.trainable_variables)
        opt_D.apply_gradients(zip(grads, D.trainable_variables))

        z = tf.random.normal((BATCH_SIZE, LATENT_DIM))
        with tf.GradientTape() as tape:
            fake = G([z, labels], training=True)
            d_fake = D([fake, labels], training=True)
            g_loss = bce(tf.ones_like(d_fake), d_fake)
        grads = tape.gradient(g_loss, G.trainable_variables)
        opt_G.apply_gradients(zip(grads, G.trainable_variables))
    return G

def gradient_penalty(C, real, fake, labels):
    eps = tf.random.uniform((tf.shape(real)[0], 1))
    interp = eps * real + (1.0 - eps) * fake
    with tf.GradientTape() as tape:
        tape.watch(interp)
        score = C([interp, labels], training=True)
    grad = tape.gradient(score, interp)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grad), axis=1) + 1e-12)
    return tf.reduce_mean((norm - 1.0)**2)

def train_cwgan_gp(X_train, y_train, minority, seed):
    set_seed(seed)
    feature_dim = X_train.shape[1]
    G = build_generator(feature_dim)
    C = build_critic(feature_dim)
    opt_C = Adam(1e-4, beta_1=0.5, beta_2=0.9)
    opt_G = Adam(1e-4, beta_1=0.5, beta_2=0.9)
    X_min = tf.convert_to_tensor(X_train[y_train == minority], tf.float32)

    for _ in range(EPOCHS_WGAN):
        for _ in range(N_CRITIC):
            idx = np.random.randint(0, len(X_min), BATCH_SIZE)
            real = tf.gather(X_min, idx)
            labels = tf.ones((BATCH_SIZE,1), tf.float32) * float(minority)
            z = tf.random.normal((BATCH_SIZE, LATENT_DIM))
            with tf.GradientTape() as tape:
                fake = G([z, labels], training=True)
                real_score = C([real, labels], training=True)
                fake_score = C([fake, labels], training=True)
                gp = gradient_penalty(C, real, fake, labels)
                c_loss = tf.reduce_mean(fake_score) - tf.reduce_mean(real_score) + GP_WEIGHT * gp
            grads = tape.gradient(c_loss, C.trainable_variables)
            opt_C.apply_gradients(zip(grads, C.trainable_variables))

        labels = tf.ones((BATCH_SIZE,1), tf.float32) * float(minority)
        z = tf.random.normal((BATCH_SIZE, LATENT_DIM))
        with tf.GradientTape() as tape:
            fake = G([z, labels], training=True)
            g_loss = -tf.reduce_mean(C([fake, labels], training=True))
        grads = tape.gradient(g_loss, G.trainable_variables)
        opt_G.apply_gradients(zip(grads, G.trainable_variables))
    return G

def generate_credit(G, n, minority, supports):
    if n <= 0:
        return np.empty((0, len(feature_names)), dtype=np.float32)
    z = np.random.normal(size=(n, LATENT_DIM)).astype(np.float32)
    labels = np.full((n,1), minority, dtype=np.float32)
    X_syn = G.predict([z, labels], verbose=0)
    return postprocess_credit(X_syn, supports)

# ---------------- MAIN EXPERIMENT ----------------
metric_rows, validation_rows, fidelity_rows = [], [], []

for run in range(N_RUNS):
    seed = BASE_SEED + run
    set_seed(seed)
    tf.keras.backend.clear_session()

    X_train, X_val, X_test, y_train, y_val, y_test, scaler = split_scale(X_raw, y, seed)
    supports = discrete_supports(X_train)
    counts = np.bincount(y_train)
    minority = int(np.argmin(counts))
    n_min, n_max = counts[minority], counts.max()

    def record(method, lam, X_fit, y_fit):
        val_df, test_df = evaluate_both(
            X_fit, y_fit, X_val, y_val, X_test, y_test, seed, minority)
        for frame, destination in [(val_df, validation_rows), (test_df, metric_rows)]:
            frame["RunID"] = run
            frame["Seed"] = seed
            frame["Method"] = method
            frame["Balancing_Strength"] = float(lam)
            destination.append(frame)

    record("Real", 0.0, X_train, y_train)

    cgan = train_cgan(X_train, y_train, minority, seed)
    cwgan = train_cwgan_gp(X_train, y_train, minority, seed)

    # Synthetic-only fixed fidelity reference versus held-out validation class.
    n_fid = max(FIDELITY_N, int(np.sum(y_val == minority)))
    fixed_fidelity = {}
    for method, G in [("CGAN", cgan), ("CWGAN-GP", cwgan)]:
        X_fid = generate_credit(G, n_fid, minority, supports)
        fixed_fidelity[method] = mean_marginal_w1(X_val[y_val == minority], X_fid)

    for lam in LAMBDA_GRID:
        _, _, _, _, target = target_minority_count(y_train, lam)
        n_add = max(0, target - n_min)

        # Random oversampling
        add_idx = (np.random.default_rng(seed + int(lam * 1000)).choice(
            np.where(y_train == minority)[0], size=n_add, replace=True)
            if n_add > 0 else np.array([], dtype=int))
        X_ros = np.vstack([X_train, X_train[add_idx]]) if n_add else X_train.copy()
        y_ros = np.concatenate([y_train, np.full(n_add, minority)]) if n_add else y_train.copy()
        record("ROS", lam, X_ros, y_ros)

        # SMOTE
        if n_add > 0:
            X_sm, y_sm = SMOTE(sampling_strategy={minority: target},
                               random_state=seed).fit_resample(X_train, y_train)
        else:
            X_sm, y_sm = X_train.copy(), y_train.copy()
        record("SMOTE", lam, X_sm, y_sm)

        # Conditional generators
        for method, G in [("CGAN", cgan), ("CWGAN-GP", cwgan)]:
            X_syn = generate_credit(G, n_add, minority, supports)
            X_aug = np.vstack([X_train, X_syn])
            y_aug = np.concatenate([y_train, np.full(n_add, minority)])
            record(method, lam, X_aug, y_aug)

            alpha_y = actual_alpha_y(n_min, n_add)
            pi_tilde_y = effective_pi_after_binary_aug(y_train, minority, n_add)
            d_hat = fixed_fidelity[method]
            fidelity_rows.append({
                "RunID": run, "Seed": seed, "Method": method,
                "Balancing_Strength": lam, "Class_ID": minority,
                "n_real_train": int(n_min), "n_generated": int(n_add),
                "alpha_y": alpha_y, "pi_tilde_y": pi_tilde_y,
                "Mean_Marginal_W1": d_hat,
                "Weighted_W1_Component": pi_tilde_y * alpha_y * d_hat
            })

    # Checkpoint after each seed, including validation and selected test results.
    metrics_ck = pd.concat(metric_rows, ignore_index=True)
    validation_ck = pd.concat(validation_rows, ignore_index=True)
    fidelity_ck = pd.DataFrame(fidelity_rows)
    metrics_ck.to_csv(f"{OUT_DIR}/credit_metrics_checkpoint.csv", index=False)
    validation_ck.to_csv(f"{OUT_DIR}/credit_validation_checkpoint.csv", index=False)
    fidelity_ck.to_csv(f"{OUT_DIR}/credit_fidelity_checkpoint.csv", index=False)
    gc.collect()

metrics = pd.concat(metric_rows, ignore_index=True)
validation = pd.concat(validation_rows, ignore_index=True)
fidelity = pd.DataFrame(fidelity_rows)

# Select independently for each run, augmentation method, and classifier.
# Validation ONLY; smaller lambda wins exact ties.
aug_validation = validation[validation["Method"] != "Real"].copy()
aug_validation = aug_validation.sort_values(
    ["RunID", "Method", "Classifier", "Macro_F1", "Balancing_Strength"],
    ascending=[True, True, True, False, True], kind="stable")
selected = aug_validation.drop_duplicates(
    ["RunID", "Method", "Classifier"], keep="first").copy()
selected = selected.rename(columns={
    "Balancing_Strength": "Selected_Lambda",
    "Macro_F1": "Val_Macro_F1", "Macro_Recall": "Val_Macro_Recall",
    "Minority_Recall": "Val_Minority_Recall",
    "Balanced_Accuracy": "Val_Balanced_Accuracy",
    "Accuracy": "Val_Accuracy"})
selected = selected.merge(
    metrics, left_on=["RunID", "Seed", "Method", "Classifier", "Selected_Lambda"],
    right_on=["RunID", "Seed", "Method", "Classifier", "Balancing_Strength"],
    how="left", validate="one_to_one")

real_selected = metrics[metrics["Method"] == "Real"].copy()
assert len(metrics) == N_RUNS * 34, f"Unexpected test rows: {len(metrics)}"
assert len(validation) == N_RUNS * 34, f"Unexpected validation rows: {len(validation)}"
assert len(selected) == N_RUNS * 8, f"Unexpected selected rows: {len(selected)}"
assert len(fidelity) == N_RUNS * 8, f"Unexpected fidelity rows: {len(fidelity)}"
assert selected["Macro_F1"].notna().all(), "Selected test merge contains missing results"

metrics.to_csv(f"{OUT_DIR}/credit_metrics_raw.csv", index=False)
validation.to_csv(f"{OUT_DIR}/credit_validation_raw.csv", index=False)
fidelity.to_csv(f"{OUT_DIR}/credit_fidelity_raw.csv", index=False)
selected.to_csv(f"{OUT_DIR}/credit_selected_test_raw.csv", index=False)
selected[["RunID", "Seed", "Method", "Classifier", "Selected_Lambda",
          "Val_Macro_F1", "Val_Macro_Recall", "Val_Minority_Recall",
          "Val_Balanced_Accuracy", "Val_Accuracy"]].to_csv(
    f"{OUT_DIR}/credit_selected_lambda_raw.csv", index=False)
real_selected.to_csv(f"{OUT_DIR}/credit_real_baseline_raw.csv", index=False)

print("Completed", N_RUNS, "runs; validation-based lambda selection verified.")
print("Outputs saved in", OUT_DIR)
display(selected.head(8))


Mounted at /content/drive
Completed 10 runs; validation-based lambda selection verified.
Outputs saved in /content/drive/MyDrive/CGA Ex/revision_protocol


,Classifier,Val_Macro_F1,Val_Macro_Recall,Val_Minority_Recall,Val_Balanced_Accuracy,Val_Accuracy,RunID,Seed,Method,Selected_Lambda,Macro_F1,Macro_Recall,Minority_Recall,Balanced_Accuracy,Accuracy,Balancing_Strength
0,MLP,0.686771,0.663059,0.384326,0.663059,0.818500,0,44,CGAN,0.50,0.672813,0.650904,0.363225,0.650904,0.811333,0.50
1,RF,0.689688,0.664878,0.384326,0.664878,0.821333,0,44,CGAN,1.00,0.678220,0.654928,0.366993,0.654928,0.815500,1.00
2,MLP,0.689774,0.671957,0.425019,0.671957,0.809667,0,44,CWGAN-GP,0.75,0.681760,0.665570,0.418237,0.665570,0.803500,0.75
3,RF,0.691258,0.666986,0.391108,0.666986,0.820833,0,44,CWGAN-GP,0.25,0.670828,0.649620,0.363225,0.649620,0.809333,0.25
4,MLP,0.684886,0.680547,0.487566,0.680547,0.788167,0,44,ROS,0.25,0.679509,0.673975,0.472494,0.673975,0.786333,0.25
5,RF,0.697904,0.676005,0.418990,0.676005,0.819333,0,44,ROS,0.25,0.679719,0.659928,0.392615,0.659928,0.809000,0.25
6,MLP,0.685764,0.673199,0.446119,0.673199,0.799833,0,44,SMOTE,0.25,0.684032,0.671640,0.443858,0.671640,0.798667,0.25
7,RF,0.697279,0.679656,0.440844,0.679656,0.812833,0,44,SMOTE,0.50,0.687244,0.670734,0.427280,0.670734,0.806500,0.50


## Reporting summaries
These summaries do not replace the run-level raw files used for inference.

In [ ]:
# Primary reporting: validation-selected test results (plus unaugmented Real).
selected_reporting = pd.concat([
    real_selected.assign(Selected_Lambda=0.0),
    selected
], ignore_index=True)
summary = (selected_reporting.groupby(["Method", "Classifier"])
           [["Macro_F1", "Macro_Recall", "Minority_Recall", "Balanced_Accuracy", "Accuracy"]]
           .agg(["mean", "std"]))
display(summary)
summary.to_csv(f"{OUT_DIR}/credit_selected_test_summary.csv")

# Descriptive sensitivity table across the complete strength grid (NOT model selection).
sensitivity = (metrics[metrics.Method != "Real"]
               .groupby(["Method", "Classifier", "Balancing_Strength"])
               [["Macro_F1", "Minority_Recall"]].agg(["mean", "std"]))
display(sensitivity)
sensitivity.to_csv(f"{OUT_DIR}/credit_sensitivity_summary.csv")


Macro_F1           Macro_Recall            \
                         mean       std         mean       std   
Method   Classifier                                              
CGAN     MLP         0.689694  0.007455     0.669338  0.008154   
         RF          0.685050  0.005222     0.661237  0.004793   
CWGAN-GP MLP         0.682762  0.007802     0.663100  0.008076   
         RF          0.681741  0.006426     0.658720  0.005455   
ROS      MLP         0.689150  0.007920     0.676674  0.008784   
         RF          0.692907  0.006691     0.673522  0.006597   
Real     MLP         0.680740  0.009703     0.658728  0.010928   
         RF          0.682572  0.006478     0.658977  0.005626   
SMOTE    MLP         0.687850  0.006895     0.677144  0.008467   
         RF          0.694405  0.005138     0.678543  0.005788   

                    Minority_Recall           Balanced_Accuracy            \
                               mean       std              mean       std   
Method   Classifier                                                         
CGAN     MLP               0.410023  0.021197          0.669338  0.008154   
         RF                0.379653  0.009882          0.661237  0.004793   
CWGAN-GP MLP               0.399472  0.020642          0.663100  0.008076   
         RF                0.377016  0.009618          0.658720  0.005455   
ROS      MLP               0.452449  0.023176          0.676674  0.008784   
         RF                0.422457  0.014348          0.673522  0.006597   
Real     MLP               0.379729  0.030815          0.658728  0.010928   
         RF                0.375283  0.010484          0.658977  0.005626   
SMOTE    MLP               0.459231  0.029997          0.677144  0.008467   
         RF                0.444461  0.017529          0.678543  0.005788   

                     Accuracy            
                         mean       std  
Method   Classifier                      
CGAN     MLP         0.813950  0.002952  
         RF          0.818267  0.002598  
CWGAN-GP MLP         0.810117  0.004118  
         RF          0.815817  0.003904  
ROS      MLP         0.801717  0.006024  
         RF          0.813533  0.003608  
Real     MLP         0.814317  0.002918  
         RF          0.817183  0.003669  
SMOTE    MLP         0.798667  0.008634  
         RF          0.809083  0.004661

Macro_F1           Minority_Recall  \
                                            mean       std            mean   
Method   Classifier Balancing_Strength                                       
CGAN     MLP        0.25                0.679310  0.010787        0.373323   
                    0.50                0.673977  0.014655        0.362095   
                    0.75                0.677139  0.009061        0.370008   
                    1.00                0.680382  0.010246        0.384099   
         RF         0.25                0.684517  0.006768        0.377845   
                    0.50                0.683088  0.005407        0.376413   
                    0.75                0.684926  0.006140        0.380030   
                    1.00                0.685270  0.005663        0.379050   
CWGAN-GP MLP        0.25                0.673499  0.006852        0.365411   
                    0.50                0.676577  0.012379        0.378146   
                    0.75                0.680201  0.009075        0.385154   
                    1.00                0.679226  0.010278        0.390656   
         RF         0.25                0.684185  0.006182        0.380859   
                    0.50                0.681318  0.006142        0.376865   
                    0.75                0.681983  0.005980        0.377167   
                    1.00                0.682845  0.004800        0.380106   
ROS      MLP        0.25                0.689150  0.007920        0.452449   
                    0.50                0.677500  0.010678        0.527807   
                    0.75                0.672364  0.007545        0.563301   
                    1.00                0.653183  0.008642        0.600452   
         RF         0.25                0.690757  0.005917        0.407687   
                    0.50                0.694005  0.005378        0.425019   
                    0.75                0.693826  0.004195        0.432027   
                    1.00                0.694446  0.005060        0.436850   
SMOTE    MLP        0.25                0.687164  0.008128        0.453203   
                    0.50                0.673666  0.006845        0.520497   
                    0.75                0.661167  0.019773        0.561341   
                    1.00                0.649531  0.014354        0.582366   
         RF         0.25                0.692448  0.005838        0.415524   
                    0.50                0.693962  0.005161        0.440090   
                    0.75                0.692535  0.004487        0.457272   
                    1.00                0.691635  0.004529        0.475207   

                                                  
                                             std  
Method   Classifier Balancing_Strength            
CGAN     MLP        0.25                0.033886  
                    0.50                0.038858  
                    0.75                0.031640  
                    1.00                0.026731  
         RF         0.25                0.011296  
                    0.50                0.008918  
                    0.75                0.009644  
                    1.00                0.010363  
CWGAN-GP MLP        0.25                0.025868  
                    0.50                0.029549  
                    0.75                0.029655  
                    1.00                0.028165  
         RF         0.25                0.011088  
                    0.50                0.009841  
                    0.75                0.009993  
                    1.00                0.008355  
ROS      MLP        0.25                0.023176  
                    0.50                0.041937  
                    0.75                0.031191  
                    1.00                0.037437  
         RF         0.25                0.009181  
                    0.50                0.010393  
                    0.75          